In [12]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import pickle
import numpy as np

from src.retriever import Retriever
from src.merger import CandidateMerger

retriever = Retriever()

In [14]:
with open("../data/parsed/text_documents.pkl", "rb") as f:
    text_documents = pickle.load(f)

with open("../data/parsed/table_documents.pkl", "rb") as f:
    table_documents = pickle.load(f)

text_embeddings = np.load("../data/embeddings/text_embeddings.npy")

table_embeddings = np.load("../data/embeddings/table_embeddings.npy")

In [15]:
query="What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement."

text_retriever = Retriever()

text_retriever.build_index(text_documents)

results = text_retriever.retrieve(
    query,
    text_embeddings,
)

In [16]:
# results = retriever.retrieve(

#     query="Operating income margin 2018",

#     documents=text_documents,

#     embeddings=text_embeddings,

#     top_k=10,

# )



In [17]:
for final_score, cosine, bm25, doc in results:

    print("=" * 80)

    print("Final :", round(final_score, 3))
    print("Cosine:", round(cosine, 3))
    print("BM25  :", round(bm25, 3))

    print(doc.metadata)

    print(doc.page_content[:500])

Final : 1.022
Cosine: 0.732
BM25  : 0.789
{'type': 'text', 'page': 91, 'label': 'text', 'window': 1}
In October 2017, 3M, via cash tender offers, repurchased $305 million aggregate principal amount of its outstanding notes. This included $110 million of its $330 million principal amount of 6.375% notes due 2028 and $195 million of its $750 million principal amount of 5.70% notes due 2037. The Company recorded an early debt extinguishment charge of approximately $96 million in the fourth quarter of 2017 within interest expense, the cash outflow for which is recorded within other financing activi
Final : 0.992
Cosine: 0.665
BM25  : 0.988
{'type': 'text', 'page': 50, 'label': 'text', 'window': 1}
Free cash flow and free cash flow conversion are not defined under U.S. generally accepted accounting principles (GAAP). Therefore, they should not be considered a substitute for income or cash flow data prepared in accordance with U.S. GAAP and may not be comparable to similarly titled measures 

In [18]:
table_retriever = Retriever()

table_retriever.build_index(table_documents)

table_results = table_retriever.retrieve(
    query,
    table_embeddings,
)

In [19]:
for final_score, cosine, bm25, doc in table_results:

    print("=" * 80)

    print("Final :", round(final_score, 3))
    print("Cosine:", round(cosine, 3))
    print("BM25  :", round(bm25, 3))

    print(doc.metadata)

    print(doc.page_content[:500])

Final : 0.868
Cosine: 0.659
BM25  : 0.571
{'page': 49, 'table': 33, 'title': 'Off-Balance Sheet Arrangements and Contractual Obligations:', 'context': "In addition to guarantees, 3M, in the normal course of business, periodically enters into agreements that require the Company to indemnify either major customers or suppliers for specific risks, such as claims for injury or property damage arising out of the use of 3M products or the negligence of 3M personnel, or claims alleging that 3M products infringe third-party patents or other intellectual property. While 3M's maximum exposure under these indemnification provisions cannot be estimated, these indemnifications are not expected to have a material impact on the Company's consolidated results of operations or financial condition.\n\nAs of December 31, 2018, the Company has not utilized special purpose entities to facilitate off-balance sheet financing arrangements. Refer to the section entitled 'Warranties/Guarantees' in Note 16 for d

In [20]:
text_results1 = text_retriever.retrieve(
    query,
    text_embeddings,
    top_k=30,
)

table_results1 = table_retriever.retrieve(
    query,
    table_embeddings,
    top_k=30,
)


In [21]:

merger = CandidateMerger()

candidates = merger.merge(
    text_results1,
    table_results1,
    top_k=40,
)

In [22]:
import sys

print(sys.executable)

d:\rag_finance\.venv\Scripts\python.exe


In [23]:
from src.reranker import Reranker

reranker = Reranker()

reranked = reranker.rerank(

    query,

    candidates,

    top_k=10,

)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [24]:
for ce, retriever_score, cosine, bm25, doc in reranked:

    print("="*80)

    print("CrossEncoder :", round(ce,3))
    print("Retriever     :", round(retriever_score,3))
    print("Cosine        :", round(cosine,3))
    print("BM25          :", round(bm25,3))

    print(doc.metadata)

    print(doc.page_content[:500])

CrossEncoder : 5.324
Retriever     : 0.906
Cosine        : 0.738
BM25          : 0.649
{'type': 'text', 'page': 8, 'label': 'text', 'window': 1}
In 2018, 3M expended approximately $27 million for capital projects related to protecting the environment. This amount excludes expenditures for remediation actions relating to existing matters caused by past operations that do not contribute to current or future revenues, which are expensed. Capital expenditures for environmental purposes have included pollution control devices - such as wastewater treatment plant improvements, scrubbers, containment structures, solvent recovery units and therm
CrossEncoder : 2.609
Retriever     : 0.893
Cosine        : 0.755
BM25          : 0.58
{'type': 'text', 'page': 45, 'label': 'text', 'window': 1}
At December 31, 2018, 3M had $3.3 billion of cash, cash equivalents and marketable securities, of which approximately $3.1 billion was held by the Company's foreign subsidiaries and approximately $160 million 

In [25]:
import sentence_transformers
import transformers
import huggingface_hub

print(sentence_transformers.__version__)
print(transformers.__version__)
print(huggingface_hub.__version__)

5.6.0
5.13.0
1.22.0


In [ ]:
# from huggingface_hub import hf_hub_download

# hf_hub_download(
#     repo_id="BAAI/bge-reranker-base",
#     filename="config.json"
# )

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

d:\rag_finance\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


'C:\\Users\\Asus\\.cache\\huggingface\\hub\\models--BAAI--bge-reranker-base\\snapshots\\2cfc18c9415c912f9d8155881c133215df768a70\\config.json'

In [47]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# model_name = "BAAI/bge-reranker-base"

# tokenizer = AutoTokenizer.from_pretrained(model_name)

# model = AutoModelForSequenceClassification.from_pretrained(model_name)

# print("Loaded successfully!")

In [26]:
from src.prompt_builder import PromptBuilder
builder = PromptBuilder()
prompt = builder.build(
    query,
    reranked,
)

In [27]:
from src.llm import LLM

In [28]:
llm = LLM()
answer = llm.generate(
    prompt,
)

In [29]:
print(answer)

Based on the provided evidence:

In 2018, 3M expended approximately $75 million for environmental purposes (excluding remediation actions) related to capital projects. This amount is expected to be approximately $75 million over the next two years (Page 8). 

63% of 3M's capital spending in 2018 was within the United States (Page 40).


In [ ]:
import ollama

response = ollama.chat(
    model="qwen3.5:4b",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print(response)

model='qwen3.5:4b' created_at='2026-07-10T15:48:38.6779146Z' done=True done_reason='length' total_duration=446558394800 load_duration=410671900 prompt_eval_count=16 prompt_eval_duration=479998000 eval_count=4080 eval_duration=445532691000 message=Message(role='assistant', content='', thinking='Thinking Process:\n\n1.  **Analyze the Request:** The user is asking me to "Say hello in one sentence." This is a direct instruction with specific constraints (one-sentence limit).\n\n2.  **Determine the Content:** I need to greet the user warmly and professionally within the single sentence constraint.\n\n3.  **Draft Options:**\n    *   Option A: Hello there!\n    *   Option B: Hi, how can I help you today?\n    *   Option C: Good morning/afternoon/evening!\n    *   Option D: It\'s great to greet you here. (Too vague)\n\n4.  **Select Best Option:** Option B is friendly and invites further interaction while adhering to the constraint. "Hello there, how can I help you today?" works well. Or simply

In [74]:
import ollama

response = ollama.chat(
    model="gemma3:4b",
    messages=[
        {
            "role": "user",
            "content": "what does mitochondria do?"
        }
    ]
)

print(response)

model='gemma3:4b' created_at='2026-07-10T16:14:14.5435641Z' done=True done_reason='stop' total_duration=43355553200 load_duration=545317500 prompt_eval_count=15 prompt_eval_duration=481499000 eval_count=571 eval_duration=42320406000 message=Message(role='assistant', content='Mitochondria are often called the "powerhouses of the cell" for a really good reason! Here\'s a breakdown of what they do:\n\n**1. Energy Production (ATP):** This is their primary and most well-known function. They produce adenosine triphosphate (ATP), which is the main energy currency used by cells to fuel all their activities – everything from muscle contraction to nerve impulse transmission.  They do this through a process called **cellular respiration**. \n\n   * **Cellular Respiration:** Essentially, mitochondria take in nutrients like glucose and oxygen, break them down, and convert that energy into ATP. There are several stages to this process:\n      * **Glycolysis** (happens partially outside the mitochond